# Fase 1 — Simulador v0: a química primeiro

Projeto LAMP · frente de Simulação, dados sintéticos e clusterização temporal · referências: LAMP-ORI-03, LAMP-S02-ESP (especificação do simulador), LAMP-S02-PROT (protocolo).

O simulador não interpola cores: ele simula a química e deixa a cor sair dela.

1. **pH ao longo do tempo** — a amplificação libera prótons e o pH cai numa curva logística (parâmetros t₀, k, A).
2. **Forma do indicador** — pela equação de Henderson-Hasselbalch, o pH define quanto do vermelho de fenol está na forma ácida (amarela) ou básica (rosa).
3. **Cor** — o espectro de absorção da mistura vira cor RGB, por colorimetria CIE ou por um sensor simplificado de três bandas.

> **Premissas desta versão (v0-provisório).** O indicador ser vermelho de fenol ainda não foi confirmado pela FMABC. As faixas de t₀, k e A são provisórias até a calibração com os tempos do artigo (fase 5). Todos os números ficam em `lampsim/config.py`. Todo resultado deste notebook é **provisório**.

Rode antes o `00_setup.ipynb` pelo menos uma vez.

## 0. Ambiente

In [ ]:
import sys
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/LAMP")
    CODIGO = BASE / "lamp-sim"
else:  # rodando fora do Colab (ex.: no computador)
    CODIGO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    BASE = CODIGO / "_dados_local"
DADOS = BASE / "lamp-sim-dados"   # fora do repositório: dados não vão para o git

assert (CODIGO / "src" / "lampsim").is_dir(), (
    f"Código não encontrado em {CODIGO}. Confira se a pasta 'lamp-sim' está em 'Meu Drive/LAMP'.")
if str(CODIGO / "src") not in sys.path:
    sys.path.insert(0, str(CODIGO / "src"))
print("Colab:", EM_COLAB)
print("Código:", CODIGO)
print("Dados :", DADOS)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lampsim.classes import NOMES_CLASSES, eixo_tempo, simular, sortear_amostra
from lampsim.config import LOTE, QUIMICA, TEMPO, VERSAO_PARAMETROS
from lampsim.cor import cor_do_indicador
from lampsim.manifesto import gerar_lote, hash_lote, montar_manifesto, salvar_manifesto
from lampsim.quimica import fracao_acida

SEMENTE_MESTRE = 2026   # mude aqui para gerar outro lote; o manifesto registra o valor
RESULTADOS = DADOS / "resultados"
LOTES = DADOS / "lotes"
for p in (RESULTADOS, LOTES):
    p.mkdir(parents=True, exist_ok=True)
PKA = QUIMICA["pka"]
LEITURA = TEMPO["leitura_nominal_min"]
print("Parâmetros:", VERSAO_PARAMETROS, "| pKa =", PKA, "| leitura nominal =", LEITURA, "min")

## 1. Do pH à cor

Curva de Henderson-Hasselbalch e a cor que cada pH produz, nas duas rotas de conversão. A transição acontece em torno do pKa: cerca de uma unidade de pH acima, quase tudo é rosa; uma abaixo, quase tudo é amarelo.

In [ ]:
ph = np.linspace(6.0, 9.5, 300)
f = fracao_acida(ph, PKA)

fig, eixos = plt.subplots(3, 1, figsize=(8, 5.5), sharex=True,
                          gridspec_kw={"height_ratios": [4, 1, 1]})
eixos[0].plot(ph, f, color="black")
eixos[0].axvline(PKA, ls="--", color="gray")
eixos[0].text(PKA + 0.05, 0.9, f"pKa = {PKA}", color="gray")
eixos[0].set_ylabel("fração ácida")
eixos[0].set_title("Vermelho de fenol: fração na forma ácida (amarela) x pH")
for ax, rota in zip(eixos[1:], ("cie", "3bandas")):
    ax.imshow(cor_do_indicador(f, rota=rota)[None], aspect="auto",
              extent=[ph[0], ph[-1], 0, 1])
    ax.set_yticks([])
    ax.set_ylabel(rota, rotation=0, ha="right", va="center")
eixos[-1].set_xlabel("pH")
fig.tight_layout()
fig.savefig(RESULTADOS / "fase1_ph_para_cor.png", dpi=120)
plt.show()

## 2. Uma amostra de cada classe

As 5 classes ópticas de `config.CLASSES`. Entre parênteses, o diagnóstico que o simulador tira da própria curva na leitura nominal, para um tubo de amostra (a classe não traz mais o diagnóstico). A linha tracejada marca a leitura nominal; a faixa cinza, a transição de cor em torno do pKa.

In [ ]:
t = eixo_tempo()
fig, ax = plt.subplots(figsize=(9, 4.5))
for n, classe in enumerate(NOMES_CLASSES):
    a = sortear_amostra(classe, SEMENTE_MESTRE + 1000 * n)  # semente diferente por classe
    s = simular(a)
    ax.plot(t, s["ph"], label=f"{classe} ({a.diagnostico})")
ax.axvline(LEITURA, ls="--", color="gray")
ax.axhspan(PKA - 0.5, PKA + 0.5, color="gray", alpha=0.12)
ax.set_xlabel("tempo (min)")
ax.set_ylabel("pH")
ax.set_title("pH ao longo da reação — uma amostra por classe")
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
fig.savefig(RESULTADOS / "fase1_ph_por_classe.png", dpi=120)
plt.show()

## 3. Fitas de cor ao longo do tempo

Cada linha é uma amostra; cada coluna, um instante. É assim que a câmera do dispositivo veria cada tubo (ainda sem as imperfeições de câmera, que entram na fase 2).

In [ ]:
N_POR_CLASSE = 8
fig, eixos = plt.subplots(len(NOMES_CLASSES), 1, figsize=(9, 9), sharex=True)
for n, (ax, classe) in enumerate(zip(eixos, NOMES_CLASSES)):
    fitas = np.stack([simular(sortear_amostra(classe, SEMENTE_MESTRE + 1000 * n + i))["rgb"]
                      for i in range(N_POR_CLASSE)])
    ax.imshow(fitas, aspect="auto", interpolation="nearest",
              extent=[t[0], t[-1], N_POR_CLASSE, 0])
    ax.axvline(LEITURA, ls="--", color="black", lw=1)
    ax.set_yticks([])
    ax.set_ylabel(classe, rotation=0, ha="right", va="center", fontsize=9)
eixos[-1].set_xlabel("tempo (min)")
fig.suptitle("Cor do tubo ao longo do tempo (rota CIE)")
fig.tight_layout()
fig.savefig(RESULTADOS / "fase1_fitas_de_cor.png", dpi=120)
plt.show()

## 4. Comparação das duas rotas de cor

A mesma amostra convertida pelas duas rotas (decisão de faseamento da LAMP-S02-ESP, Seção 2.4). As duas mudam de cor no mesmo sentido; a de três bandas é menos saturada, como um sensor barato.

In [ ]:
amostra = sortear_amostra("positivo_perto_lod", SEMENTE_MESTRE)
fig, eixos = plt.subplots(2, 1, figsize=(9, 1.8), sharex=True)
for ax, rota in zip(eixos, ("cie", "3bandas")):
    ax.imshow(simular(amostra, rota=rota)["rgb"][None], aspect="auto",
              extent=[t[0], t[-1], 0, 1])
    ax.set_yticks([])
    ax.set_ylabel(rota, rotation=0, ha="right", va="center")
eixos[-1].set_xlabel("tempo (min)")
fig.tight_layout()
plt.show()

## 5. Lote v1-mini, com manifesto

Gera 20 amostras por grupo de `config.LOTE` (grupo, classe óptica, papel do tubo) a partir da semente-mestre e grava no Drive:

- `manifesto_<hash>.json` — semente, versões e parâmetros sorteados de cada amostra (este arquivo pode ir para o git);
- `trajetorias_<hash>.npz` — tempo, pH, fração ácida e cor de cada amostra (fica só no Drive).

O hash identifica o lote: a mesma semente com a mesma versão de parâmetros dá sempre o mesmo hash.

In [ ]:
N = 20
lote = gerar_lote(N, SEMENTE_MESTRE)
manifesto = montar_manifesto(lote, SEMENTE_MESTRE, N)
caminho_manifesto = salvar_manifesto(manifesto, LOTES)

sims = [simular(a) for a in lote]
caminho_traj = LOTES / f"trajetorias_{manifesto['hash_lote']}.npz"
np.savez_compressed(
    caminho_traj,
    t=t,
    ids=np.array([a.id for a in lote]),
    classe=np.array([a.classe for a in lote]),
    papel=np.array([a.papel for a in lote]),
    diagnostico=np.array([a.diagnostico for a in lote]),
    ph=np.stack([s["ph"] for s in sims]),
    fracao_acida=np.stack([s["fracao_acida"] for s in sims]),
    rgb=np.stack([s["rgb"] for s in sims]),
)
print("amostras:", len(lote), "| hash do lote:", manifesto["hash_lote"])
print("manifesto :", caminho_manifesto)
print("trajetórias:", caminho_traj)

# reprodutibilidade: gerar de novo com a mesma semente dá o mesmo lote
assert hash_lote(gerar_lote(N, SEMENTE_MESTRE)) == manifesto["hash_lote"]
print("Reprodutível: mesma semente, mesmo hash.")

## 6. Conferência por classe

Duas leituras por grupo do lote, na leitura nominal e no fim da reação:

- **passou da virada** — fração ácida > 0,5 (pH abaixo do pKa; tubo já laranja ou amarelo);
- **amarelo** — fração ácida > 0,9 (pH abaixo de ~7; a cor final de um positivo).

A última coluna conta os diagnósticos do grupo, tirados da curva de cada tubo (positivo, negativo, controle_ok ou invalido).

O esperado: positivos fortes amarelos já na leitura nominal; negativo e controle negativo sempre rosados (o controle sai controle_ok); o controle contaminado vira tarde e sai inválido quando já virou na leitura; a falha de reação já começa ácida e sai inválida; e o positivo perto do LoD **dividido** — muitos passam da virada mas só parte chega ao amarelo. É a região de fronteira, onde a decisão é difícil e onde o protocolo concentra o teste.

In [ ]:
from collections import Counter

i_leitura = int(round(LEITURA / TEMPO["passo_min"]))
rotulo = f"{int(LEITURA)} min"
print(f"{'':42}{'passou da virada':>22}{'amarelo':>20}")
print(f"{'grupo':22} {'papel':18} {rotulo:>10} {'30 min':>10} {rotulo:>10} {'30 min':>10}  diagnóstico")
for grupo, _classe, papel in LOTE:
    do_grupo = [(s, a) for s, a in zip(sims, lote) if a.id.rsplit("-", 1)[0] == grupo]
    f = np.stack([s["fracao_acida"] for s, _ in do_grupo])
    diagnosticos = Counter(a.diagnostico for _, a in do_grupo)
    texto_diag = ", ".join(f"{d} {n}" for d, n in sorted(diagnosticos.items()))
    print(f"{grupo:22} {papel:18} "
          f"{(f[:, i_leitura] > 0.5).mean():10.0%} {(f[:, -1] > 0.5).mean():10.0%} "
          f"{(f[:, i_leitura] > 0.9).mean():10.0%} {(f[:, -1] > 0.9).mean():10.0%}  {texto_diag}")

## Fase 1 pronta quando

- [ ] Os testes do `00_setup` passam (química, cor, classes e manifesto).
- [ ] A cor muda num só sentido (rosa → laranja → amarelo) e os gráficos acima fazem sentido.
- [ ] O lote v1-mini e o manifesto estão em `Meu Drive/LAMP/lamp-sim-dados/lotes`, e o hash se repete.
- [ ] As faixas de `lampsim/config.py` foram conferidas com as tabelas da LAMP-S02-ESP.

**Em aberto (não bloqueiam a fase 2):** confirmação do indicador pela FMABC; espectro medido do indicador para substituir as bandas gaussianas; calibração de t₀ e k com os tempos do artigo (fase 5).

Próximo passo: fase 2 — imagens e imperfeições de câmera.